# 03 · İlk CNN Modeli

**Amaç:** Kendi tasarladığımız küçük CNN'i kurmak, eğitmek ve **doğrulama kümesinde** ne kadar iyi olduğunu görmek. Bu model sonraki defterlerin karşılaştırma noktası olacak.

| Bölüm | Ne yapılıyor? |
|---|---|
| 1. Kurulum ve ayarlar | GPU kontrolü, eğitim ayarları |
| 2. Veriyi yükleme | 02'de işlenen veri ve bölme Drive'dan okunur |
| 3. DataLoader | Veriyi modele küçük gruplar halinde vermek |
| 4. Model | CNN'in katmanları ve parametre sayısı |
| 5. Kayıp fonksiyonu ve optimizer | Hata nasıl ölçülüyor, ağırlıklar nasıl güncelleniyor |
| 6. Eğitim | Eğitim döngüsü ve erken durdurma |
| 7. Öğrenme eğrileri | Model öğreniyor mu, ezberliyor mu? |
| 8. Doğrulama sonuçları | Karar eşiği, ölçütler, karmaşıklık matrisi |
| 9. Kaydetme | Model ve sonuçlar Drive'a |

**Colab'da T4 GPU çalışma zamanıyla çalıştır:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **T4 GPU**. Süre yaklaşık 15–25 dakika.

**Test kümesi bu defterde hiç kullanılmıyor.** Test sonucu en sonda, değerlendirme defterinde bir kez hesaplanacak.

In [ ]:
# --- Kurulum: hem Google Colab'da hem kendi bilgisayarında çalışır ---
import os, sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")                     # işlenmiş veri Drive'da saklanır
    REPO = Path("/content/tornado-cnn")
    if not REPO.exists():
        try:                                          # repo private ise: Colab Secrets'taki token
            token = userdata.get("GITHUB_TOKEN") + "@"
        except Exception:                             # repo public ise token gerekmez
            token = ""
        !git clone -q https://{token}github.com/baevv/tornado-cnn.git {REPO}
    else:
        !git -C {REPO} pull -q
    %pip install -q h5netcdf h5py
    os.chdir(REPO)
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(REPO))
# src/ klasöründen daha önce yüklenmiş kodu bellekten sil: git pull ile gelen yeni kod
# bir sonraki import'ta kullanılsın (oturumu yeniden başlatmaya gerek kalmasın).
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

print("Colab" if IN_COLAB else "Yerel", "| repo:", REPO)

## 1. Kurulum ve ayarlar

Eğitim ayarları `configs/train.json` dosyasından okunuyor. Sonraki defterlerdeki denemeler bu ayarları değiştirip aynı kodu kullanacak.

| Ayar | Değer | Anlamı |
|---|---|---|
| `batch_size` | 64 | Her adımda modele verilen görüntü sayısı (3. hafta: mini batch) |
| `optimizer` | adam | Ağırlıkları güncelleyen yöntem (4. hafta) |
| `lr` | 0,001 | Öğrenme oranı: her güncellemede ne kadar büyük adım atılacağı |
| `weight_decay` | 0 | L2 cezası (5. hafta); ilk modelde kapalı, denemelerde açılacak |
| `epochs` | 20 | En fazla tur sayısı; erken durdurma çoğunlukla daha önce bitirir |
| `patience` | 4 | Doğrulama AUC'si 4 tur üst üste iyileşmezse eğitim durur |
| `dropout` | 0,3 | Eğitimde nöronların %30'u rastgele kapatılır; ezberi azaltır (5. hafta) |
| `weighting` | pos_weight | Sınıf dengesizliği için kayıp ağırlığı (5. bölümde) |
| `seed` | 42 | Aynı ayarla her çalıştırma aynı sonucu versin diye |

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from src import paths, train
from src import model as cnn

cfg_data = paths.load_config()
with open(REPO / "configs" / "train.json") as f:
    cfg = json.load(f)

device = train.get_device()
if device.type == "cuda":
    print("cihaz: GPU |", torch.cuda.get_device_name(0))
else:
    print("UYARI: GPU yok, eğitim çok yavaş olur. Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU")
train.set_seed(cfg["seed"])
cfg

## 2. Veriyi yükleme

* 02'de işlenen `.npz` dosyaları Drive'dan Colab'ın kendi diskine kopyalanır. Eğitim sırasında veri hızlı yerel diskten okunur.
* Veri belleğe **sıkıştırılmış haliyle** (`uint8`, piksel başına 1 bayt) alınır. Ondalıklı sayıya çevirme her grup için ayrı yapılır; hepsini birden çevirseydik 4 kat yer kaplardı ve belleğe sığmazdı.
* Bölme **Drive'daki `splits.csv`'den** okunur, yeniden hesaplanmaz. Herkes birebir aynı kümelerle çalışır.
* Test kümesi yüklenmez.

**Hızlı deneme modu:** `HIZLI_DENEME = True` yaparsan sadece 2017 kullanılır ve defter birkaç dakikada biter. Kodu denemek için; sonuçları gerçek sonuç değildir.

In [ ]:
HIZLI_DENEME = False      # True: sadece 2017 ile hızlı deneme

years = [2017] if HIZLI_DENEME else cfg_data["years"]
splits = train.load_splits(paths.data_dir())
local_dir = Path("/content/data_local") if IN_COLAB else None
data, variables = train.load_data(paths.processed_dir(), splits, years, which=("train", "val"), local_dir=local_dir)

summary = pd.DataFrame({
    s: {"örnek": len(d["y"]), "hortum": int(d["y"].sum()), "hortum_%": round(100 * d["y"].mean(), 1),
        "bellek (GB)": round(d["X"].nbytes / 1e9, 2)}
    for s, d in data.items()
}).T
print("yıllar:", years, "| kanallar:", variables, "| bir örnek:", data["train"]["X"].shape[1:])
summary

## 3. DataLoader

**DataLoader nedir?** Veriyi modele **küçük gruplar (batch)** halinde veren yapı. 49 bin görüntüyü tek seferde vermek yerine 64'erli gruplar halinde veriyor; ağırlıklar her gruptan sonra güncelleniyor (3. hafta: mini batch).

* **Eğitim** verisi her turda **karıştırılıyor**: model örneklerin sırasını ezberlemesin, her grupta farklı örnekler olsun.
* **Doğrulama** verisi karıştırılmıyor; sadece ölçüm için kullanılıyor.
* Her grup GPU'ya geçerken `uint8`'den **[-1, 1]** aralığına çevriliyor ve boş pikseller dolduruluyor (DBZ → −1, VEL → 0, WIDTH → −1).

In [ ]:
w_train, pos_weight = train.class_weights(data["train"]["meta"], cfg["weighting"])
train_loader, val_loader = train.make_loaders(data, cfg, w_train)

xb, yb, wb = next(iter(train_loader))
x = train.decode_batch(xb.to(device), variables)
print(f"bir grup: {list(x.shape)} | değerler [{x.min():.2f}, {x.max():.2f}] | bu grupta {int(yb.sum())} hortum")
print(f"eğitimde {len(train_loader)} grup, doğrulamada {len(val_loader)} grup")

## 4. Model

**Katmanlar ve görevleri** (kod: `src/model.py`):

| Katman | Görevi | Derste |
|---|---|---|
| **Evrişim (Conv 3×3)** | Küçük bir filtre görüntünün üzerinde kayar, her konumda "çarp, topla, bias ekle" yapar. Çıktısı, o şeklin görüntünün neresinde olduğunu gösteren bir özellik haritası. | 6. hafta |
| **ReLU** | Negatif değerleri sıfırlar; ağa doğrusal olmayan davranış kazandırır. | 2. hafta |
| **Max havuzlama (2×2)** | Görüntüyü yarıya küçültür, her bölgenin en büyük değerini tutar. | 6. hafta |
| **Global max havuzlama** | Her özellik haritasından tek bir sayı alır: "bu şekil görüntünün herhangi bir yerinde var mı?" Hortum nerede olursa olsun aynı karar verilir ve parametre sayısı küçük kalır. | 6. hafta |
| **Tam bağlı (Dense)** | Bulunan özelliklerden kararı verir; aslında küçük bir MLP. | 2–3. hafta |
| **Dropout** | Eğitimde nöronların bir kısmını rastgele kapatır; model tek bir özelliğe aşırı güvenmez. Tahmin sırasında kapalıdır. | 5. hafta |

Son katmanda **sigmoid yok**: Çıktı bir "logit" (herhangi bir sayı). Sigmoid kayıp fonksiyonunun içinde uygulanıyor (5. bölüm); olasılık gerektiğinde ayrıca uygulanıyor.

**Neden bu mimari?** Görüntü verisi olduğu için CNN; küçük tutuldu çünkü ~49 bin eğitim görüntüsü için milyonlarca parametreli bir model kolayca ezberler. Filtre sayısı derinleştikçe artıyor (16 → 128): ilk katmanlar basit geçişleri, son katmanlar bunların birleşimini (hız çifti gibi) öğreniyor.

Aşağıdaki tabloda her katmanın çıktı boyutu ve parametre sayısı var. Parametreler hem **6. haftadaki formülle elle** hem de **PyTorch'un saydığıyla** hesaplanıyor.

In [ ]:
net = cnn.TornadoCNN(in_channels=len(variables), dropout=cfg["dropout"]).to(device)
table = cnn.layer_table(net, input_shape=tuple(data["train"]["X"].shape[1:]))
total_torch, total_manual = cnn.count_params(net), int(table["parametre (elle)"].sum())
print(f"toplam parametre: PyTorch {total_torch:,} | elle {total_manual:,} | aynı mı: {total_torch == total_manual}")
table

## 5. Kayıp fonksiyonu ve optimizer

**Kayıp fonksiyonu: `BCEWithLogitsLoss`** = sigmoid + ikili çapraz entropi, tek bir işlemde (1. hafta: sigmoid, 3. hafta: çapraz entropi). İkisini tek işlemde yapmak sayısal olarak daha kararlı. Model "hortum yok" derken gerçekte hortum varsa (ya da tersi) kayıp büyür.

**`pos_weight`: dengesizlik için ağırlık.** Eğitimde her hortuma yaklaşık 11–12 hortumsuz örnek düşüyor. Ağırlık olmazsa model her şeye "hortum yok" deyip düşük kayıp alabilir. `pos_weight = hortumsuz sayısı / hortum sayısı` ile hortumu kaçırmak o kadar kat daha fazla cezalandırılıyor.

**Optimizer: Adam** (4. hafta): Momentum (geçmiş gradyanların yönünü hatırlar) ile RMSProp'u (her ağırlığa ayrı adım boyu) birleştirir. Öğrenme oranı 0,001.

In [ ]:
n_pos = int(data["train"]["y"].sum())
n_neg = len(data["train"]["y"]) - n_pos
print(f"ağırlık türü: {cfg['weighting']}")
print(f"pos_weight = {n_neg:,} / {n_pos:,} = {pos_weight:.2f}")
print(f"optimizer: {cfg['optimizer']} | öğrenme oranı: {cfg['lr']}")

## 6. Eğitim

Her grupta (batch) dört adım yapılıyor (3. hafta, bir eğitim adımı):

```
1. İleri yayılım:  model 64 görüntüye bakar, her biri için bir logit üretir
2. Kayıp:          tahminler gerçek etiketlerle karşılaştırılır
3. Geri yayılım:   loss.backward()  → her ağırlığın gradyanı hesaplanır
4. Güncelleme:     optimizer.step() → Adam ağırlıkları günceller
```

Bütün eğitim verisi bir kez görülünce bir **tur (epoch)** biter. Her turun sonunda model **doğrulama kümesinde** ölçülür: kayıp, ROC-AUC ve PR-AUC.

* **Erken durdurma:** Doğrulama AUC'si 4 tur üst üste iyileşmezse eğitim durur ve **en iyi turdaki ağırlıklara** geri dönülür. Aşırı öğrenmeye karşı önlem (5. hafta).
* **Kayıt:** Her tur Drive'a kaydedilir. Bağlantı koparsa bu hücreyi tekrar çalıştır; kaldığı turdan devam eder.
* ★ işareti o ana kadarki en iyi turu gösterir.

In [ ]:
OUT = paths.models_dir("03_baseline_hizli" if HIZLI_DENEME else "03_baseline")
print("kayıt klasörü:", OUT)
history = train.fit(net, train_loader, val_loader, cfg, variables, OUT, pos_weight, device)

## 7. Öğrenme eğrileri

* **Sol grafik, kayıp:** İki eğri birlikte düşüyorsa model öğreniyor. **Eğitim kaybı düşmeye devam ederken doğrulama kaybı yükselmeye başlarsa** model eğitim verisini ezberlemeye başlamış demektir (aşırı öğrenme).
* **Sağ grafik, doğrulama AUC ve PR-AUC:** Modelin hortumu ne kadar iyi ayırt ettiği. Kesikli çizgi en iyi turu gösteriyor; o turun ağırlıkları kullanılıyor.

PR-AUC, ROC-AUC'den daha düşük çıkar; bu normal. Nadir sınıfta (hortum) daha hassas bir ölçüt: rastgele bir modelin PR-AUC'si hortum oranı kadardır (~0,08).

In [ ]:
best = int(history.loc[history["doğrulama AUC"].idxmax(), "tur"])
fig, axes = plt.subplots(1, 2, figsize=(13, 4), layout="constrained")
axes[0].plot(history["tur"], history["eğitim kaybı"], "o-", label="eğitim")
axes[0].plot(history["tur"], history["doğrulama kaybı"], "o-", label="doğrulama")
axes[0].set_title("Kayıp")
axes[1].plot(history["tur"], history["doğrulama AUC"], "o-", label="ROC-AUC")
axes[1].plot(history["tur"], history["doğrulama PR-AUC"], "o-", label="PR-AUC")
axes[1].set_title("Doğrulama ölçütleri")
for ax in axes:
    ax.axvline(best, color="gray", ls="--", label=f"en iyi tur ({best})")
    ax.set_xlabel("tur (epoch)")
    ax.legend()
plt.show()
history.round(4)

## 8. Doğrulama sonuçları

**Karar eşiği:** Model bir olasılık veriyor; "hortum var" demek için bir eşik gerekiyor. 0,5 dengesiz veride iyi bir seçim değil. Eşiği **doğrulama kümesinde F1'i en yüksek yapan** değer olarak seçiyoruz. Aynı eşik test kümesinde de aynen kullanılacak (test kümesine bakarak eşik seçmiyoruz).

**Ölçütler:**

| Ölçüt | Soru |
|---|---|
| Doğruluk | Tahminlerin yüzde kaçı doğru? (dengesiz veride yanıltıcı) |
| Recall (yakalama) | Gerçek hortumların yüzde kaçını bulduk? |
| Precision (isabet) | "Hortum" dediklerimizin yüzde kaçı gerçekten hortum? |
| F1 | Recall ile precision'ın dengesi |
| ROC-AUC, PR-AUC | Eşikten bağımsız ayırt etme gücü |

**Karşılaştırma noktası:** Her örneğe "hortum yok" diyen bir model. Doğruluğu yüksek görünür ama hiçbir hortum bulamaz.

In [ ]:
y_val = data["val"]["y"]
p_val, _ = train.predict(net, val_loader, variables, device, pos_weight)
threshold = train.choose_threshold(y_val, p_val)
CHOSEN = f"Bizim CNN (seçilen eşik {threshold:.3f})"

rows = {
    "Her şeye 'hortum yok'": train.metrics(y_val, np.zeros_like(p_val), 0.5),
    "Bizim CNN (eşik 0.5)": train.metrics(y_val, p_val, 0.5),
    CHOSEN: train.metrics(y_val, p_val, threshold),
}
results_val = pd.DataFrame(rows).T
print(f"seçilen eşik: {threshold:.3f}")
results_val.round(3)

**Karmaşıklık matrisi** (seçilen eşikle, doğrulama kümesi):

* **Sol üst:** Doğru "hortum yok" · **Sağ üst:** Yanlış alarm
* **Sol alt:** Kaçırılan hortum · **Sağ alt:** Yakalanan hortum

In [ ]:
m = rows[CHOSEN]
cm = np.array([[m["TN"], m["FP"]], [m["FN"], m["TP"]]])
fig, ax = plt.subplots(figsize=(4.5, 4), layout="constrained")
ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", fontsize=14,
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xticks([0, 1], ["hortum yok", "hortum var"])
ax.set_yticks([0, 1], ["hortum yok", "hortum var"])
ax.set_xlabel("model tahmini")
ax.set_ylabel("gerçek")
ax.set_title("Karmaşıklık matrisi (doğrulama)")
plt.show()

## 9. Kaydetme

Drive'a (`tornado-cnn/models/03_baseline/`) kaydedilenler:

* `best.pt`: en iyi turdaki model ağırlıkları
* `last.pt`: son durum (kaldığı yerden devam etmek için)
* `history.json`: tur tur kayıp ve AUC değerleri
* `results.json`: ayarlar, seçilen eşik, doğrulama sonuçları

Sonraki defterlerdeki denemeler bu sonuçla karşılaştırılacak; değerlendirme, Grad-CAM ve arayüz defterleri de bu modeli kullanacak.

In [ ]:
train.save_results(
    OUT, config=cfg, years=years, variables=variables, pos_weight=pos_weight,
    best_epoch=best, threshold=threshold,
    validation=results_val.loc[CHOSEN].to_dict(),
)
for f in sorted(OUT.iterdir()):
    print(f"{f.name:15s} {f.stat().st_size / 1e6:6.2f} MB")